# Simulació de Propagació d'Incendi Forestal — Sessió 2

## Descripció

En aquesta sessió modelitzem la propagació d'un incendi forestal amb un **autòmat cel·lular m:n-CA^k** representable amb SDL.  
Treballem amb **dues capes principals** carregades des de fitxers en format IDRISI32 i IDRISI31 vectorial, i una tercera capa que registra la propagació dinàmica.

### Objectiu
Simular com s'estén un incendi a través d'un terreny heterogeni, considerant la humitat i la vegetació com a variables principals, amb la possibilitat d'incorporar l'efecte del vent.

---

### Assumpcions extretes de l'enunciat

1. La **vegetació** representa les **hores de combustió** d'una cel·la (ex: 10 → la cel·la triga 10 passos a cremar-se completament).
2. La **humitat** representa les **hores que han de passar** des que una cel·la adjacent crema fins que el foc s'inicia a la cel·la.
3. Un cop eliminada la humitat, la vegetació comença a cremar i la cel·la **ja pot propagar el foc** als seus veïns.
4. El model ha de tenir **com a mínim dues capes que interactuïn** (vegetació i humitat) i una **tercera capa de propagació** amb tres estats: cremat, en procés de cremar-se i pendent de cremar-se.

### Assumpcions pròpies de disseny

5. La propagació és **determinista**: si una cel·la té un veí cremant i ha eliminat la seva humitat, el foc es propaga sempre. La variabilitat del sistema ja ve donada pels valors heterogenis de vegetació i humitat de cada cel·la.
6. La propagació segueix un **veïnatge de Moore (8 veïns)**: permet la propagació en diagonal, més realista que Von Neumann.
7. El vent s'implementa com un **factor de prioritat direccional**: afavoreix la propagació cap a la seva direcció, però no la bloqueja en les altres.

---
## 1. Importacions i constants

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import os

# ── Estats de les cel·les ──────────────────────────────────────
class CellState:
    EMPTY   = 0   # pendent de cremar (vegetació sana)
    MOIST   = 1   # escalfant-se (eliminant humitat)
    BURNING = 2   # cremant activament → propaga foc als veïns
    BURNED  = 3   # ja cremat (cendra)

# ── Mapa de codis de terreny (Initialize.img) → humitat base (hores) ──
TERRAIN_HUMIDITY = {
    "LL": 0,
    "CT": 1, "TE": 1,
    "GR": 2, "CC": 2, "CAT": 2,
    "AA": 3, "BS": 3,
    "BN": 4,
    "BC": 5,
}

# ── Mapa de polígon ID (vegetation.vec) → hores de combustió ──
POLYGON_VEGETATION = {
    1: 10,
    2: 5,
    20: 3,
}
DEFAULT_VEGETATION = 4

# ── Colors i llegenda ──────────────────────────────────────────
CMAP_COLORS = ["#639922", "#F4C0D1", "#EF9F27", "#444441"]
CMAP = mcolors.ListedColormap(CMAP_COLORS)
NORM = mcolors.BoundaryNorm([0, 1, 2, 3, 4], CMAP.N)
LEGEND_PATCHES = [
    mpatches.Patch(color=CMAP_COLORS[0], label="Vegetació (sana)"),
    mpatches.Patch(color=CMAP_COLORS[1], label="Escalfant-se"),
    mpatches.Patch(color=CMAP_COLORS[2], label="En flames"),
    mpatches.Patch(color=CMAP_COLORS[3], label="Cremada"),
]

print("Importacions completades.")

---
## 2. Lectura de fitxers IDRISI32 — Capa d'Humitat

Els fitxers `Initialize.doc` (capçalera) i `Initialize.img` (dades) segueixen el format IDRISI32 ASCII.
- El `.doc` conté metadades: mides, sistema de referència, etc.
- El `.img` conté codis de terreny que es mapegen a valors d'humitat (hores).

In [ ]:
def parse_idrisi_doc(path):
    """Llegeix una capçalera IDRISI32 (.doc) i retorna un dict clau:valor."""
    meta = {}
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if ":" in line:
                key, _, val = line.partition(":")
                meta[key.strip().lower()] = val.strip()
    return meta


def parse_idrisi_img_ascii(path):
    """Llegeix un fitxer de dades IDRISI32 ASCII (.img). Retorna llista de strings."""
    values = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            v = line.strip()
            if v:
                values.append(v)
    return values


def build_humidity_grid(img_values, rows, cols):
    """Construeix la quadrícula de humitat des de codis ASCII, repetint cíclicament."""
    grid = np.zeros((rows, cols), dtype=float)
    n = len(img_values)
    for r in range(rows):
        for c in range(cols):
            code = img_values[(r * cols + c) % n].upper()
            grid[r, c] = TERRAIN_HUMIDITY.get(code, 1)
    return grid


BASE_DIR = "dades"
meta     = parse_idrisi_doc(os.path.join(BASE_DIR, "Initialize.doc"))
img_vals = parse_idrisi_img_ascii(os.path.join(BASE_DIR, "Initialize.img"))

print("Metadades Initialize.doc:")
for k, v in meta.items():
    print(f"   {k:20s}: {v}")
print(f"\nValors únics a Initialize.img: {sorted(set(img_vals))}")

---
## 3. Lectura de fitxers IDRISI31 vectorials — Capa de Vegetació

Els fitxers `vegetation.dvc` (capçalera) i `vegetation.vec` (polígons) segueixen el format IDRISI31.
Cada polígon té un `id` que es mapeja a les hores de combustió. S'usa **ray-casting** per rasteritzar-los.

In [ ]:
def parse_vec(path):
    """Llegeix un fitxer .vec IDRISI31 i retorna una llista de polígons."""
    polygons = []
    with open(path, "r", errors="ignore") as f:
        lines = [l.strip() for l in f if l.strip()]
    i = 0
    while i < len(lines):
        parts = lines[i].split()
        if len(parts) < 2:
            i += 1; continue
        try:
            poly_id = int(parts[0])
            int(parts[1])
        except ValueError:
            i += 1; continue
        i += 1
        points = []
        while i < len(lines):
            xy = lines[i].split(); i += 1
            if len(xy) < 2: continue
            x, y = float(xy[0]), float(xy[1])
            if x == 0.0 and y == 0.0: break
            points.append((x, y))
        if points:
            polygons.append({"id": poly_id, "points": points})
    return polygons


def point_in_polygon(px, py, pts):
    """Ray-casting: retorna True si (px, py) és dins del polígon."""
    n, inside, j = len(pts), False, len(pts) - 1
    for i in range(n):
        xi, yi = pts[i]; xj, yj = pts[j]
        if ((yi > py) != (yj > py)) and (px < (xj-xi)*(py-yi)/(yj-yi+1e-12)+xi):
            inside = not inside
        j = i
    return inside


def build_vegetation_grid(polygons, rows, cols, min_x, max_x, min_y, max_y):
    """Rasteritza els polígons vectorials a una graella rows×cols via ray-casting."""
    grid = np.full((rows, cols), DEFAULT_VEGETATION, dtype=float)
    for r in range(rows):
        for c in range(cols):
            px = (c + 0.5) / cols * (max_x - min_x) + min_x
            py = (r + 0.5) / rows * (max_y - min_y) + min_y
            for poly in polygons:
                if point_in_polygon(px, py, poly["points"]):
                    grid[r, c] = POLYGON_VEGETATION.get(poly["id"], DEFAULT_VEGETATION)
                    break
    return grid


veg_meta = parse_idrisi_doc(os.path.join(BASE_DIR, "vegetation.dvc"))
polygons = parse_vec(os.path.join(BASE_DIR, "vegetation.vec"))

print(f"Polígons llegits: {len(polygons)}")
for p in polygons:
    print(f"   id={p['id']:3d} → {POLYGON_VEGETATION.get(p['id'], DEFAULT_VEGETATION):2d}h combustió, "
          f"{len(p['points'])} punts")

---
## 4. Construcció de les capes

Llegim la mida directament de la capçalera `Initialize.doc` i construïm les dues capes d'entrada.

In [ ]:
# ══════════════════════════════════════════════════════
#  PARÀMETRES DE LA SIMULACIÓ (modifica aquí!)
# ══════════════════════════════════════════════════════
IGNITE_CELLS = [(5, 5)]   # focus d'ignició (fila, col) — afegeix més si cal
WIND_DIR     = (0, 1)     # (Δfila, Δcol): (0,1)=est, (1,0)=sud, (0,0)=sense vent
MAX_STEPS    = 80         # passos màxims de simulació
# ══════════════════════════════════════════════════════

# Mida llegida del fitxer de capçalera
ROWS = int(meta.get("rows", 20))
COLS = int(meta.get("columns", 20))

hum_grid = build_humidity_grid(img_vals, ROWS, COLS)

min_x = float(veg_meta.get("min. x", 0))
max_x = float(veg_meta.get("max. x", 100))
min_y = float(veg_meta.get("min. y", 0))
max_y = float(veg_meta.get("max. y", 100))
veg_grid = build_vegetation_grid(polygons, ROWS, COLS, min_x, max_x, min_y, max_y)

print(f"Graella {ROWS}×{COLS} construïda")
print(f"   Humitat    min/max : {hum_grid.min():.1f} / {hum_grid.max():.1f} hores")
print(f"   Vegetació  min/max : {veg_grid.min():.1f} / {veg_grid.max():.1f} hores")
print(f"   Focus d'ignició    : {IGNITE_CELLS}")

---
## 5. Visualització de les capes inicials

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Capes d'entrada — Estat inicial", fontsize=14, fontweight='bold')

im0 = axes[0].imshow(hum_grid, cmap="Blues", vmin=0, vmax=5)
axes[0].set_title("Capa d'Humitat (hores)")
axes[0].set_xlabel("Columna"); axes[0].set_ylabel("Fila")
plt.colorbar(im0, ax=axes[0], shrink=0.85, label="Hores")
for (r, c) in IGNITE_CELLS:
    axes[0].plot(c, r, 'r*', markersize=12)

im1 = axes[1].imshow(veg_grid, cmap="Greens", vmin=0, vmax=12)
axes[1].set_title("Capa de Vegetació (hores de combustió)")
axes[1].set_xlabel("Columna"); axes[1].set_ylabel("Fila")
plt.colorbar(im1, ax=axes[1], shrink=0.85, label="Hores")
for (r, c) in IGNITE_CELLS:
    axes[1].plot(c, r, 'r*', markersize=12)

plt.tight_layout()
plt.show()

---
## 6. Model m:n-CA^k — Autòmat Cel·lular

### Regles de transició

| Estat actual | Condició | Estat següent |
|---|---|---|
| `EMPTY` | Veí cremant + humitat > 0 | `MOIST` (comença a escalfar-se) |
| `EMPTY` | Veí cremant + humitat = 0 | `BURNING` (s'encén directament) |
| `MOIST` | `hum_timer` ≥ `humitat_inicial` | `BURNING` (humitat esgotada) |
| `BURNING` | `burn_timer` ≥ `vegetació_inicial` | `BURNED` (vegetació consumida) |

La propagació és **determinista**: si la condició es compleix, el canvi d'estat és sempre cert.  
El vent modifica l'**ordre de propagació**, afavorint els veïns en la seva direcció.

In [ ]:
class WildfireCA:
    """
    Model m:n-CA^k per a la propagació d'un incendi forestal.

    Capes:
      state       : CellState de cada cel·la
      hum_init    : humitat inicial (hores)
      veg_init    : vegetació inicial (hores de combustió)
      burn_timer  : comptador de passos cremant
      hum_timer   : comptador de passos escalfant-se
    """

    def __init__(self, humidity_grid, vegetation_grid, wind_dir=(0, 0)):
        self.rows, self.cols = humidity_grid.shape
        self.hum_init  = humidity_grid.copy().astype(float)
        self.veg_init  = vegetation_grid.copy().astype(float)
        self.wind_dir  = wind_dir
        self.max_veg   = max(1.0, float(np.max(vegetation_grid)))
        self.reset()

    def reset(self):
        self.state      = np.full((self.rows, self.cols), CellState.EMPTY, dtype=int)
        self.burn_timer = np.zeros((self.rows, self.cols), dtype=float)
        self.hum_timer  = np.zeros((self.rows, self.cols), dtype=float)
        self.veg_cur    = self.veg_init.copy()
        self.step       = 0
        self.history    = []

    def ignite(self, row, col):
        """Encén manualment la cel·la (row, col)."""
        if 0 <= row < self.rows and 0 <= col < self.cols:
            self.state[row, col]      = CellState.BURNING
            self.burn_timer[row, col] = 0
            print(f"[{row:3d},{col:3d}]  veg={self.veg_init[row,col]:.0f}h  "
                  f"hum={self.hum_init[row,col]:.1f}h")

    def _sorted_neighbors(self, r, c):
        """
        Retorna els 8 veïns ordenats: primer els que estan en la direcció del vent,
        garantint que es processen abans (propagació determinista però dirigida).
        """
        wr, wc = self.wind_dir
        neighbors = []
        for dr in [-1, 0, 1]:
            for dc in [-1, 0, 1]:
                if dr == 0 and dc == 0:
                    continue
                nr, nc = r + dr, c + dc
                if 0 <= nr < self.rows and 0 <= nc < self.cols:
                    align = dr * wr + dc * wc
                    neighbors.append((align, nr, nc))
        neighbors.sort(key=lambda x: -x[0])
        return [(nr, nc) for (_, nr, nc) in neighbors]

    def advance(self):
        """Executa un pas de l'autòmat cel·lular (determinista)."""
        new_state      = self.state.copy()
        new_burn_timer = self.burn_timer.copy()
        new_hum_timer  = self.hum_timer.copy()
        new_veg_cur    = self.veg_cur.copy()

        for r in range(self.rows):
            for c in range(self.cols):
                s = self.state[r, c]

                if s == CellState.BURNING:
                    new_burn_timer[r, c] += 1
                    new_veg_cur[r, c]     = max(0.0, self.veg_init[r, c] - new_burn_timer[r, c])
                    if new_burn_timer[r, c] >= self.veg_init[r, c]:
                        new_state[r, c] = CellState.BURNED
                    else:
                        for (nr, nc) in self._sorted_neighbors(r, c):
                            if self.state[nr, nc] != CellState.EMPTY:
                                continue
                            if self.hum_init[nr, nc] > 0:
                                new_state[nr, nc]     = CellState.MOIST
                                new_hum_timer[nr, nc] = 0
                            else:
                                new_state[nr, nc]      = CellState.BURNING
                                new_burn_timer[nr, nc] = 0

                elif s == CellState.MOIST:
                    new_hum_timer[r, c] += 1
                    if new_hum_timer[r, c] >= self.hum_init[r, c]:
                        new_state[r, c]      = CellState.BURNING
                        new_burn_timer[r, c] = 0

        self.state      = new_state
        self.burn_timer = new_burn_timer
        self.hum_timer  = new_hum_timer
        self.veg_cur    = new_veg_cur
        self.step      += 1
        self.history.append(self.state.copy())

    def render_rgb(self):
        """Imatge RGB amb gradient de color continu per estat i intensitat."""
        img = np.zeros((self.rows, self.cols, 3), dtype=np.uint8)
        for r in range(self.rows):
            for c in range(self.cols):
                s = self.state[r, c]
                if s == CellState.EMPTY:
                    ratio = min(1.0, self.veg_cur[r, c] / self.max_veg)
                    img[r, c] = [0, int(80 + ratio * 150), 0]
                elif s == CellState.MOIST:
                    img[r, c] = [244, 192, 209]
                elif s == CellState.BURNING:
                    ratio = 1.0 - min(1.0, self.veg_cur[r, c] / max(1, self.veg_init[r, c]))
                    img[r, c] = [255, int(200 * (1 - ratio)), 0]
                else:  # BURNED
                    img[r, c] = [50, 50, 50]
        return img

    def stats(self):
        total   = self.rows * self.cols
        burning = int(np.sum(self.state == CellState.BURNING))
        moist   = int(np.sum(self.state == CellState.MOIST))
        burned  = int(np.sum(self.state == CellState.BURNED))
        safe    = total - burning - moist - burned
        return {"step": self.step, "burning": burning, "moist": moist,
                "burned": burned, "safe": safe, "total": total}

    def is_active(self):
        return (np.any(self.state == CellState.BURNING) or
                np.any(self.state == CellState.MOIST))

print("Classe WildfireCA definida (model determinista amb vent direccional).")

---
## 7. Execució de la simulació

In [ ]:
ca = WildfireCA(hum_grid, veg_grid, wind_dir=WIND_DIR)

print(f"Encenent {len(IGNITE_CELLS)} focus d'ignició:")
for (r, c) in IGNITE_CELLS:
    ca.ignite(r, c)

stats_history = []

print(f"\n{'Pas':>5}  {'Flames':>9}  {'Escalfant':>11}  {'Cremat':>9}  {'Sa':>7}  {'%Cremat':>8}")
print("-" * 62)

for _ in range(MAX_STEPS):
    if not ca.is_active():
        print("  → Foc extingit.")
        break
    ca.advance()
    s = ca.stats()
    stats_history.append(s)
    pct = s["burned"] / s["total"] * 100
    print(f"{s['step']:>5}  {s['burning']:>9}  {s['moist']:>11}  "
          f"{s['burned']:>9}  {s['safe']:>7}  {pct:>7.1f}%")

print("-" * 62)
sf = ca.stats()
print(f"\nResultat final — Pas {sf['step']}:")
print(f"   Cremat : {sf['burned']:5d} cel·les ({sf['burned']/sf['total']*100:.1f}%)")
print(f"   Sa     : {sf['safe']:5d} cel·les ({sf['safe']/sf['total']*100:.1f}%)")

---
## 8. Visualització del resultat final

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f"Simulació d'Incendi Forestal — Resultat Final (Pas {ca.step})",
             fontsize=14, fontweight='bold')

im0 = axes[0].imshow(hum_grid, cmap="Blues", vmin=0, vmax=5)
axes[0].set_title("Capa d'Humitat (inicial)")
axes[0].set_xlabel("Columna"); axes[0].set_ylabel("Fila")
plt.colorbar(im0, ax=axes[0], shrink=0.8, label="Hores")

im1 = axes[1].imshow(veg_grid, cmap="Greens", vmin=0, vmax=12)
axes[1].set_title("Capa de Vegetació (inicial)")
axes[1].set_xlabel("Columna"); axes[1].set_ylabel("Fila")
plt.colorbar(im1, ax=axes[1], shrink=0.8, label="Hores")

axes[2].imshow(ca.render_rgb())
axes[2].set_title("Propagació final (gradient de color)")
axes[2].set_xlabel("Columna"); axes[2].set_ylabel("Fila")
axes[2].legend(handles=LEGEND_PATCHES, loc="upper right", fontsize=9, framealpha=0.85)
for (r, c) in IGNITE_CELLS:
    axes[2].plot(c, r, 'w*', markersize=14, markeredgecolor='black', markeredgewidth=0.5)

plt.tight_layout()
plt.savefig("output/resultat_final.png", dpi=130, bbox_inches="tight")
plt.show()

---
## 9. Evolució temporal de la simulació

In [ ]:
if stats_history:
    steps   = [s["step"]    for s in stats_history]
    burning = [s["burning"] for s in stats_history]
    moist   = [s["moist"]   for s in stats_history]
    burned  = [s["burned"]  for s in stats_history]
    safe    = [s["safe"]    for s in stats_history]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle("Evolució temporal de la simulació", fontsize=13, fontweight='bold')

    ax1.plot(steps, burning, color="#EF9F27", lw=2, label="En flames")
    ax1.plot(steps, moist,   color="#F4C0D1", lw=2, label="Escalfant-se")
    ax1.plot(steps, burned,  color="#444441", lw=2, label="Cremat")
    ax1.plot(steps, safe,    color="#639922", lw=2, label="Sa")
    ax1.set_xlabel("Pas de temps"); ax1.set_ylabel("Nombre de cel·les")
    ax1.set_title("Cel·les per estat"); ax1.legend(fontsize=9); ax1.grid(True, alpha=0.3)

    pct_burned = [b / ca.rows / ca.cols * 100 for b in burned]
    ax2.fill_between(steps, pct_burned, color="#444441", alpha=0.7)
    ax2.set_xlabel("Pas de temps"); ax2.set_ylabel("% del territori cremat")
    ax2.set_title("Superfície cremada acumulada")
    ax2.set_ylim(0, 100); ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("outputs/evolucio_temporal.png", dpi=130, bbox_inches="tight")
    plt.show()
    print("Figura guardada: evolucio_temporal.png")

---
## 10. Fotogrames de la propagació

In [ ]:
if ca.history:
    n_frames = len(ca.history)
    indices  = np.linspace(0, n_frames - 1, min(12, n_frames), dtype=int)

    ncols = 4
    nrows = int(np.ceil(len(indices) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.8, nrows * 3.5))
    axes = axes.flatten()
    fig.suptitle("Propagació de l'incendi — fotogrames clau", fontsize=13, fontweight='bold')

    COLOR_MAP = {
        CellState.EMPTY:   [60, 160, 60],
        CellState.MOIST:   [244, 192, 209],
        CellState.BURNING: [239, 159, 39],
        CellState.BURNED:  [50, 50, 50],
    }

    for i, idx in enumerate(indices):
        frame = ca.history[idx]
        img = np.zeros((ROWS, COLS, 3), dtype=np.uint8)
        for state, color in COLOR_MAP.items():
            img[frame == state] = color
        axes[i].imshow(img)
        axes[i].set_title(f"Pas {idx + 1}", fontsize=9)
        axes[i].axis('off')
        for (r, c) in IGNITE_CELLS:
            axes[i].plot(c, r, 'w*', markersize=8)

    for j in range(len(indices), len(axes)):
        axes[j].axis('off')

    fig.legend(handles=LEGEND_PATCHES, loc='lower center',
               ncol=4, fontsize=9, bbox_to_anchor=(0.5, -0.02))
    plt.tight_layout()
    plt.savefig("outputs/fotogrames_propagacio.png",
                dpi=130, bbox_inches="tight")
    plt.show()

---
## 11. Experiments: comparació de condicions de vent

Comparem l'impacte de la direcció del vent sobre la propagació (model determinista).

In [ ]:
conditions = [
    {"label": "Sense vent",   "wind": (0, 0)},
    {"label": "Vent Est (→)", "wind": (0, 1)},
    {"label": "Vent Sud (↓)", "wind": (1, 0)},
    {"label": "Vent SE (↘)",  "wind": (1, 1)},
]

fig, axes = plt.subplots(1, len(conditions), figsize=(5.5 * len(conditions), 5))
fig.suptitle("Comparació: efecte del vent sobre la propagació (determinista)",
             fontsize=13, fontweight='bold')

for ax, cond in zip(axes, conditions):
    ca_exp = WildfireCA(hum_grid, veg_grid, wind_dir=cond["wind"])
    for (r, c) in IGNITE_CELLS:
        ca_exp.ignite(r, c)
    for _ in range(MAX_STEPS):
        if not ca_exp.is_active(): break
        ca_exp.advance()
    s = ca_exp.stats()
    pct = s['burned'] / s['total'] * 100

    ax.imshow(ca_exp.render_rgb())
    for (r, c) in IGNITE_CELLS:
        ax.plot(c, r, 'w*', markersize=10, markeredgecolor='k', markeredgewidth=0.5)
    ax.set_title(f"{cond['label']}\n{pct:.1f}% cremat ({s['step']} passos)", fontsize=10)
    ax.set_xlabel("Columna"); ax.set_ylabel("Fila")

fig.legend(handles=LEGEND_PATCHES, loc='lower center',
           ncol=4, fontsize=9, bbox_to_anchor=(0.5, -0.04))
plt.tight_layout()
plt.savefig("outputs/comparacio_vent.png", dpi=130, bbox_inches="tight")
plt.show()

---
## 12. Conclusions

### Model implementat
S'ha implementat un **autòmat cel·lular m:n-CA^k** per simular la propagació d'un incendi forestal sobre una graella bidimensional de **{ROWS}×{COLS} cel·les** llegida dels fitxers IDRISI32. El model utilitza **tres capes**:

1. **Capa d'humitat** (de `Initialize.doc/img`): retarda l'inici del foc a cada cel·la. Les cel·les amb humitat 0 s'encenen directament; les altres passen primer per l'estat `MOIST`.
2. **Capa de vegetació** (de `vegetation.dvc/vec`, rasteritzada via ray-casting): determina durant quants passos crema cada cel·la i per tant quan pot propagar foc als veïns.
3. **Capa de propagació**: conté l'estat dinàmic de cada cel·la: sana (`EMPTY`) → escalfant-se (`MOIST`) → en flames (`BURNING`) → cremada (`BURNED`).

### Observacions
- El model és **determinista**: donat un estat inicial, l'evolució és sempre la mateixa. No hi ha aleatorietat; la variabilitat espacial ve dels valors heterogenis de les capes d'entrada.
- La **humitat** actua com a retard: les zones amb valors alts (ex: `BN`, `BC`) triguen més a encendre's, cosa que pot desviar el front del foc.
- La **vegetació densa** (polígons amb 10h) crema llarg temps i actua com a focus persistent que facilita la propagació als veïns.
- El **vent** altera la seqüència de propagació: en direcció est, els veïns de la dreta es processen primer, fent que el front avanci asimètricament.

### Limitacions i possibles millores
- La topografia (pendent) no es considera, malgrat ser molt rellevant en incendis reals.
- El vent és un vector constant; un camp espacialment variable seria més realista.
- La mida de la graella (20×20) és petita; amb dades reals caldria una resolució major.

---
## 13. Ús de la Intel·ligència Artificial

Per al desenvolupament d'aquesta pràctica s'ha fet servir **Claude (Anthropic)** com a eina de suport.

L'ús de la IA ha estat útil principalment en tres àmbits:

1. **Estructuració del codi**: ajuda a organitzar el notebook en seccions clares i a definir l'arquitectura de la classe `WildfireCA`.
2. **Revisió crítica**: comparant la nostra implementació amb la d'un alumne de l'any anterior, la IA va identificar elements que faltaven i, igualment important, va ajudar a descartar elements que no corresponien a l'enunciat (probabilitat estocàstica, zones d'aigua, generació de dades no requerides).
3. **Depuració**: va ajudar a corregir errors en la lectura dels formats IDRISI32/31 i en la lògica de ray-casting per als polígons vectorials.

En general, la IA ha accelerat el procés i ha permès centrar l'esforç en les decisions de disseny del model. Les decisions clau (determinisme del CA, tractament de la humitat i vegetació, implementació del vent) s'han pres de forma independent, usant la IA com a eina de verificació i implementació, no com a substitut del raonament propi.